# Day 4: Hands-On Practice Tasks – Advanced LangGraph Workflows

## Overview
This notebook covers three advanced hands-on tasks:
1. **Exercise 1**: Expense Calculator Agent (Cyclic Graph – Loops)
2. **Exercise 2**: Travel Planner (Memory-Based Bot)
3. **Exercise 3**: Access Control Approval System (Human-in-the-Loop)

Each exercise builds on LangGraph concepts and demonstrates real-world AI agent patterns.

### Key Learning Outcomes
- ✓ Understanding cyclic agent flows and tool loops
- ✓ Implementing memory retention across conversation turns
- ✓ Building human-in-the-loop approval workflows
- ✓ Tracking agent state and messages
- ✓ Multi-step tool execution and chaining

In [1]:
# Install required dependencies
import subprocess
import sys

# Install packages
packages = ["langgraph", "langchain", "langchain-anthropic"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All dependencies installed successfully!")

✓ All dependencies installed successfully!


---

## Exercise 1: Expense Calculator Agent (Cyclic Graph – Loops)

### Objective
Build a cyclic agent that performs multi-step calculations using tools.

### Problem Statement
Create an agent that can:
- Add expenses
- Apply tax (e.g., GST 18%)
- Split total among people

### Tools to Implement
- `add_expenses(a, b)` – Add two amounts
- `apply_tax(amount, tax_percent)` – Apply tax percentage
- `split_bill(amount, people)` – Split bill among people

### Sample Inputs
- "What is the total of 1200 and 800?"
- "Add 2000 and 1500, apply 18% tax, and split among 3 people."

### Expected Learning Outcomes
- ✓ Understanding cyclic agent flow (agent ↔ tools loop)
- ✓ Multi-step tool execution and chaining
- ✓ Tracking state through multiple tool calls
- ✓ Practical financial computation use-case

In [2]:
## API Credentials Configuration

import os

# Configure Custom API Credentials (Same as Day 3)
# Replace with your actual API key
API_KEY = os.environ.get("KEY") # Change this to your actual key
BASE_URL = "https://llmgw-wp.tekstac.com"
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# Set environment variables
os.environ["KEY"] = API_KEY

# Verify API configuration
try:
    if API_KEY == "your-api-key-here":
        print("✗ Please update API_KEY with your actual key")
    else:
        print("✓ API credentials configured successfully!")
        print(f"✓ Model: {MODEL_ID}")
        print(f"✓ Base URL: {BASE_URL}")
        print("✓ Ready to use Claude for all exercises")
except Exception as e:
    print(f"✗ Error configuring API: {e}")

✓ API credentials configured successfully!
✓ Model: global.anthropic.claude-haiku-4-5-20251001-v1:0
✓ Base URL: https://llmgw-wp.tekstac.com
✓ Ready to use Claude for all exercises


In [3]:
### Exercise 1: Implementation - With Claude LLM

import os
import re
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

# Initialize Claude LLM with custom endpoint
llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    max_tokens=300
)

# Define the agent state
class ExpenseState(TypedDict):
    user_input: str
    amounts: list
    total: float
    tax_amount: float
    final_amount: float
    split_result: dict
    messages: list

# Tool 1: Add expenses
def add_expenses(a: float, b: float) -> float:
    """Add two expenses and return total"""
    result = a + b
    print(f"🔧 Tool: add_expenses({a}, {b}) = {result}")
    return result

# Tool 2: Apply tax
def apply_tax(amount: float, tax_percent: float) -> dict:
    """Apply tax to amount and return both tax and total"""
    tax = amount * (tax_percent / 100)
    total = amount + tax
    print(f"🔧 Tool: apply_tax({amount}, {tax_percent}%) = Tax: {tax:.2f}, Total: {total:.2f}")
    return {"tax": tax, "total": total}

# Tool 3: Split bill
def split_bill(amount: float, people: int) -> dict:
    """Split bill among people"""
    per_person = amount / people
    print(f"🔧 Tool: split_bill({amount}, {people}) = {per_person:.2f} per person")
    return {"per_person": per_person, "total_people": people}

# Agent Node: Use Claude to parse input and extract amounts/operations
def expense_agent(state: ExpenseState) -> ExpenseState:
    """Use Claude LLM to intelligently parse user input"""
    user_input = state["user_input"]
    messages = state.get("messages", [])
    
    messages.append(f"User: {user_input}")
    print(f"\n📊 Agent Processing: {user_input}")
    
    # Use Claude to extract amounts and identify operations
    prompt = f"""Extract the following from this user request:
1. Amounts (as numbers)
2. Operations (add, tax, split)
3. Specific values (tax percentage, number of people)

User request: {user_input}

Respond in this format:
AMOUNTS: [number1, number2, ...]
OPERATIONS: [operation1, operation2, ...]
TAX_PERCENT: [percentage or None]
SPLIT_PEOPLE: [number or None]"""
    
    response = llm.invoke(prompt)
    agent_response = response.content.strip()
    print(f"🤖 Claude Analysis:\n{agent_response}")
    messages.append(f"Claude Analysis: Parsed operations and amounts")
    
    # Extract amounts from Claude's response
    amounts_match = re.search(r'AMOUNTS:\s*\[(.*?)\]', agent_response)
    if amounts_match:
        try:
            amounts_str = amounts_match.group(1)
            state["amounts"] = [float(x.strip()) for x in amounts_str.split(',') if x.strip()]
        except:
            pass
    
    state["messages"] = messages
    return state

# Tool execution nodes
def add_expenses_node(state: ExpenseState) -> ExpenseState:
    """Execute add_expenses tool"""
    if len(state["amounts"]) >= 2:
        a, b = state["amounts"][0], state["amounts"][1]
        state["total"] = add_expenses(a, b)
        state["messages"].append(f"Addition Result: {a} + {b} = {state['total']}")
    
    return state

def apply_tax_node(state: ExpenseState) -> ExpenseState:
    """Execute apply_tax tool"""
    if state["total"] > 0 and "tax" in state["user_input"].lower():
        result = apply_tax(state["total"], 18)
        state["tax_amount"] = result["tax"]
        state["final_amount"] = result["total"]
        state["messages"].append(f"Tax Applied: Tax Amount: ₹{result['tax']:.2f}, Final Amount: ₹{result['total']:.2f}")
    else:
        state["final_amount"] = state["total"]
    
    return state

def split_bill_node(state: ExpenseState) -> ExpenseState:
    """Execute split_bill tool"""
    people_match = re.search(r'split.*?(\d+)', state["user_input"], re.IGNORECASE)
    
    if people_match and state["final_amount"] > 0:
        people = int(people_match.group(1))
        result = split_bill(state["final_amount"], people)
        state["split_result"] = result
        state["messages"].append(f"Split Result: ₹{result['per_person']:.2f} per person ({result['total_people']} people)")
    
    return state

# Build the agent graph
graph_builder = StateGraph(ExpenseState)

# Add nodes
graph_builder.add_node("agent", expense_agent)
graph_builder.add_node("add_expenses", add_expenses_node)
graph_builder.add_node("apply_tax", apply_tax_node)
graph_builder.add_node("split_bill", split_bill_node)

# Define edges to create cyclic flow
graph_builder.add_edge(START, "agent")
graph_builder.add_edge("agent", "add_expenses")
graph_builder.add_edge("add_expenses", "apply_tax")
graph_builder.add_edge("apply_tax", "split_bill")
graph_builder.add_edge("split_bill", END)

# Compile the graph
expense_agent_graph = graph_builder.compile()

print("✓ Expense Calculator Agent Graph built with Claude LLM!")
print("\nAgent Flow:")
print("START → agent (Claude parsing) → add_expenses → apply_tax → split_bill → END")

/home/ubuntu/Desktop/LangChain/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Expense Calculator Agent Graph built with Claude LLM!

Agent Flow:
START → agent (Claude parsing) → add_expenses → apply_tax → split_bill → END


In [4]:
### Exercise 1: Test Cases

# Test Case 1: Simple addition
print("=" * 60)
print("TEST CASE 1: Simple Addition")
print("=" * 60)

input_state_1 = {
    "user_input": "What is the total of 1200 and 800?",
    "amounts": [],
    "total": 0,
    "tax_amount": 0,
    "final_amount": 0,
    "split_result": {},
    "messages": []
}

result_1 = expense_agent_graph.invoke(input_state_1)

print("\n📋 Final State:")
print(f"  Total: {result_1['total']}")
print(f"  Messages:")
for msg in result_1['messages']:
    print(f"    - {msg}")

# Test Case 2: Full workflow - Add, Apply Tax, Split
print("\n" + "=" * 60)
print("TEST CASE 2: Full Workflow (Add → Tax → Split)")
print("=" * 60)

input_state_2 = {
    "user_input": "Add 2000 and 1500, apply 18% tax, and split among 3 people.",
    "amounts": [],
    "total": 0,
    "tax_amount": 0,
    "final_amount": 0,
    "split_result": {},
    "messages": []
}

result_2 = expense_agent_graph.invoke(input_state_2)

print("\n📋 Final State:")
print(f"  Initial Amounts: {result_2['amounts']}")
print(f"  Total: ₹{result_2['total']:.2f}")
print(f"  Tax Amount (18%): ₹{result_2['tax_amount']:.2f}")
print(f"  Final Amount: ₹{result_2['final_amount']:.2f}")
print(f"  Split Result: ₹{result_2['split_result'].get('per_person', 0):.2f} per person")
print(f"\n  Agent Messages:")
for msg in result_2['messages']:
    print(f"    ✓ {msg}")

print("\n✅ Exercise 1 Complete - Cyclic Agent Flow Demonstrated!")

TEST CASE 1: Simple Addition

📊 Agent Processing: What is the total of 1200 and 800?
🤖 Claude Analysis:
AMOUNTS: [1200, 800]
OPERATIONS: [add]
TAX_PERCENT: None
SPLIT_PEOPLE: None
🔧 Tool: add_expenses(1200.0, 800.0) = 2000.0

📋 Final State:
  Total: 2000.0
  Messages:
    - User: What is the total of 1200 and 800?
    - Claude Analysis: Parsed operations and amounts
    - Addition Result: 1200.0 + 800.0 = 2000.0

TEST CASE 2: Full Workflow (Add → Tax → Split)

📊 Agent Processing: Add 2000 and 1500, apply 18% tax, and split among 3 people.
🤖 Claude Analysis:
AMOUNTS: [2000, 1500]
OPERATIONS: [add, tax, split]
TAX_PERCENT: [18]
SPLIT_PEOPLE: [3]
🔧 Tool: add_expenses(2000.0, 1500.0) = 3500.0
🔧 Tool: apply_tax(3500.0, 18%) = Tax: 630.00, Total: 4130.00
🔧 Tool: split_bill(4130.0, 3) = 1376.67 per person

📋 Final State:
  Initial Amounts: [2000.0, 1500.0]
  Total: ₹3500.00
  Tax Amount (18%): ₹630.00
  Final Amount: ₹4130.00
  Split Result: ₹1376.67 per person

  Agent Messages:
    ✓ User: 

---

## Exercise 2: Travel Planner (Memory-Based Bot)

### Objective
Build a chatbot that remembers user details across multiple conversation turns.

### Problem Statement
Create a travel assistant that remembers:
- Destination
- Travel dates
- Preferences (budget/luxury, food, etc.)

### Sample Flow
- Turn 1: "I am planning a trip to Bali."
- Turn 2: "I prefer budget hotels."
- Turn 3: "My trip is in December."
- Turn 4: "Summarise my travel plan."

### Expected Output
"You are planning a trip to Bali in December and prefer budget hotels."

### Expected Learning Outcomes
- ✓ Using MemorySaver for context persistence
- ✓ Using thread_id for session tracking
- ✓ Context retention across multiple calls
- ✓ Building stateful chatbot interactions

In [5]:
### Exercise 2: Implementation - Travel Planner with Claude & Memory

import os
import re
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_anthropic import ChatAnthropic

# Initialize Claude LLM for travel planning
travel_llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    max_tokens=250
)

# Define the travel planner state
class TravelState(TypedDict):
    messages: list
    destination: str
    travel_dates: str
    preferences: list
    summary: str

# Node 1: Use Claude to extract travel information
def travel_info_extractor(state: TravelState) -> TravelState:
    """Use Claude LLM to intelligently extract travel information"""
    messages = state.get("messages", [])
    all_messages = " ".join(messages)
    
    # Use Claude to extract travel information
    prompt = f"""Extract travel planning information from this conversation.
Find and extract:
1. Destination (city/country)
2. Travel dates (month/season)
3. Preferences (budget/luxury, food, activities, etc.)

Conversation:
{all_messages}

Respond in this format:
DESTINATION: [destination or "Not mentioned"]
TRAVEL_DATES: [month/date or "Not mentioned"]
PREFERENCES: [preference1, preference2, ...]"""
    
    response = travel_llm.invoke(prompt)
    claude_output = response.content.strip()
    print(f"🤖 Claude Analysis:\n{claude_output}\n")
    
    # Extract destination
    dest_match = re.search(r'DESTINATION:\s*(.+?)(?=\n|$)', claude_output)
    if dest_match:
        dest = dest_match.group(1).strip()
        if dest and dest != "Not mentioned":
            state["destination"] = dest
            print(f"✓ Destination found: {dest}")
    
    # Extract travel dates
    dates_match = re.search(r'TRAVEL_DATES:\s*(.+?)(?=\n|$)', claude_output)
    if dates_match:
        dates = dates_match.group(1).strip()
        if dates and dates != "Not mentioned":
            state["travel_dates"] = dates
            print(f"✓ Travel dates found: {dates}")
    
    # Extract preferences
    prefs_match = re.search(r'PREFERENCES:\s*\[(.*?)\]', claude_output)
    if prefs_match:
        prefs_str = prefs_match.group(1)
        new_prefs = [p.strip() for p in prefs_str.split(',') if p.strip() and p.strip() != ""]
        for pref in new_prefs:
            if pref not in state.get("preferences", []):
                state["preferences"].append(pref)
                print(f"✓ Preference found: {pref}")
    
    return state

# Node 2: Use Claude to generate travel summary
def travel_summarizer(state: TravelState) -> TravelState:
    """Use Claude LLM to generate a personalized travel summary"""
    destination = state.get("destination", "")
    travel_dates = state.get("travel_dates", "")
    preferences = state.get("preferences", [])
    
    # Use Claude to generate a nice summary
    prompt = f"""Create a friendly travel plan summary based on:
Destination: {destination if destination else "Not yet mentioned"}
Travel dates: {travel_dates if travel_dates else "Not yet mentioned"}
Preferences: {', '.join(preferences) if preferences else "Not yet mentioned"}

Write a one-line summary that feels natural and helpful."""
    
    response = travel_llm.invoke(prompt)
    summary = response.content.strip()
    state["summary"] = summary
    print(f"✓ Summary generated: {summary}")
    
    return state

# Build the travel planner graph with memory
travel_graph_builder = StateGraph(TravelState)

# Add nodes
travel_graph_builder.add_node("extractor", travel_info_extractor)
travel_graph_builder.add_node("summarizer", travel_summarizer)

# Define edges
travel_graph_builder.add_edge(START, "extractor")
travel_graph_builder.add_edge("extractor", "summarizer")
travel_graph_builder.add_edge("summarizer", END)

# Create memory saver for persistence
memory = MemorySaver()

# Compile with checkpointing
travel_planner_graph = travel_graph_builder.compile(checkpointer=memory)

print("✓ Travel Planner Agent built with Claude LLM & MemorySaver!")
print("✓ Uses Claude for intelligent information extraction")
print("✓ Uses MemorySaver for context persistence across turns")

✓ Travel Planner Agent built with Claude LLM & MemorySaver!
✓ Uses Claude for intelligent information extraction
✓ Uses MemorySaver for context persistence across turns


In [6]:
### Exercise 2: Test Cases - Multi-Turn Conversation with Memory

# Create a thread_id for session tracking
thread_id = "travel-session-001"
config = {"configurable": {"thread_id": thread_id}}

print("=" * 70)
print("TRAVEL PLANNER - MULTI-TURN CONVERSATION WITH MEMORY")
print("=" * 70)
print(f"Session ID (thread_id): {thread_id}\n")

# State to accumulate across turns
accumulated_state = {
    "messages": [],
    "destination": "",
    "travel_dates": "",
    "preferences": [],
    "summary": ""
}

# Turn 1: User mentions destination
print("🔄 TURN 1: Destination")
print("-" * 70)
turn1_input = {
    "messages": ["I am planning a trip to Bali."],
    "destination": "",
    "travel_dates": "",
    "preferences": [],
    "summary": ""
}

result1 = travel_planner_graph.invoke(turn1_input, config)
accumulated_state["destination"] = result1["destination"]
accumulated_state["messages"].extend(turn1_input["messages"])
print(f"User: {turn1_input['messages'][0]}")
print(f"Stored Destination: {result1['destination']}\n")

# Turn 2: User mentions preferences
print("🔄 TURN 2: Preferences")
print("-" * 70)
turn2_input = {
    "messages": accumulated_state["messages"] + ["I prefer budget hotels."],
    "destination": accumulated_state["destination"],
    "travel_dates": accumulated_state["travel_dates"],
    "preferences": accumulated_state["preferences"],
    "summary": ""
}

result2 = travel_planner_graph.invoke(turn2_input, config)
accumulated_state["preferences"] = result2["preferences"]
accumulated_state["messages"] = turn2_input["messages"]
print(f"User: I prefer budget hotels.")
print(f"Stored Preferences: {result2['preferences']}\n")

# Turn 3: User mentions travel dates
print("🔄 TURN 3: Travel Dates")
print("-" * 70)
turn3_input = {
    "messages": accumulated_state["messages"] + ["My trip is in December."],
    "destination": accumulated_state["destination"],
    "travel_dates": accumulated_state["travel_dates"],
    "preferences": accumulated_state["preferences"],
    "summary": ""
}

result3 = travel_planner_graph.invoke(turn3_input, config)
accumulated_state["travel_dates"] = result3["travel_dates"]
accumulated_state["messages"] = turn3_input["messages"]
print(f"User: My trip is in December.")
print(f"Stored Travel Dates: {result3['travel_dates']}\n")

# Turn 4: User asks for summary
print("🔄 TURN 4: Summary Request")
print("-" * 70)
turn4_input = {
    "messages": accumulated_state["messages"] + ["Summarise my travel plan."],
    "destination": accumulated_state["destination"],
    "travel_dates": accumulated_state["travel_dates"],
    "preferences": accumulated_state["preferences"],
    "summary": ""
}

result4 = travel_planner_graph.invoke(turn4_input, config)
print(f"User: Summarise my travel plan.")
print(f"\n🎯 FINAL SUMMARY:")
print(f"   {result4['summary']}")

print("\n" + "=" * 70)
print("✅ Exercise 2 Complete - Memory Retention Demonstrated!")
print("=" * 70)
print(f"\n📊 Memory Flow:")
print(f"  Turn 1 → Destination stored: Bali")
print(f"  Turn 2 → Preferences added: budget hotels")
print(f"  Turn 3 → Travel dates added: December")
print(f"  Turn 4 → All info combined into summary")

TRAVEL PLANNER - MULTI-TURN CONVERSATION WITH MEMORY
Session ID (thread_id): travel-session-001

🔄 TURN 1: Destination
----------------------------------------------------------------------
🤖 Claude Analysis:
DESTINATION: Bali
TRAVEL_DATES: Not mentioned
PREFERENCES: Not mentioned

✓ Destination found: Bali
✓ Summary generated: # Your Bali Adventure Awaits! 🌴

Once you share your travel dates and what you're hoping to experience—whether that's beaches, temples, rice terraces, or vibrant nightlife—I can help you craft the perfect itinerary for your Bali getaway!
User: I am planning a trip to Bali.
Stored Destination: Bali

🔄 TURN 2: Preferences
----------------------------------------------------------------------
🤖 Claude Analysis:
DESTINATION: Bali
TRAVEL_DATES: Not mentioned
PREFERENCES: Budget hotels

✓ Destination found: Bali
✓ Summary generated: # Your Bali Adventure Awaits! 🌴

Here's what we know so far: You're dreaming of Bali! To help you plan the perfect trip, I'd love to know

---

## Exercise 3: Access Control Approval System (Human-in-the-Loop)

### Objective
Build an agent that pauses for human approval before executing actions.

### Problem Statement
Create an IT access control system that:
- Grants system access
- Revokes access
- Requires manager approval before execution

### Tools to Implement
- `grant_access(user, system)` – Grant system access to user
- `revoke_access(user, system)` – Revoke system access from user

### Sample Scenario
"Give database admin access to user 'Kiran'"

### Expected Flow
1. Agent proposes tool call
2. System pauses (HITL - Human-in-the-Loop)
3. Ask: "Approve? (yes/no)"
4. If approved → execute tool
5. If rejected → stop and log rejection

### Expected Learning Outcomes
- ✓ Using interrupt_before=["tools"] for pausing
- ✓ Implementing Human-in-the-Loop workflows
- ✓ Real-world enterprise approval systems
- ✓ State management with approval tracking

In [7]:
### Exercise 3: Implementation - Access Control with Claude & HITL

import os
import re
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

# Initialize Claude LLM for access control analysis
access_llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    max_tokens=250
)

# Define the access control state
class AccessControlState(TypedDict):
    request: str
    user: str
    system: str
    action: str
    reasoning: str
    approval_status: str
    execution_result: str
    messages: list

# Tool 1: Grant access
def grant_access(user: str, system: str) -> dict:
    """Grant system access to user"""
    result = {
        "status": "granted",
        "user": user,
        "system": system,
        "timestamp": "2024-01-15T10:30:00Z"
    }
    print(f"✓ Tool Executed: grant_access({user}, {system})")
    print(f"  Status: {result['status']}")
    return result

# Tool 2: Revoke access
def revoke_access(user: str, system: str) -> dict:
    """Revoke system access from user"""
    result = {
        "status": "revoked",
        "user": user,
        "system": system,
        "timestamp": "2024-01-15T10:30:00Z"
    }
    print(f"✓ Tool Executed: revoke_access({user}, {system})")
    print(f"  Status: {result['status']}")
    return result

# Agent Node: Use Claude to parse access request
def access_control_agent(state: AccessControlState) -> AccessControlState:
    """Use Claude to intelligently parse access request"""
    request = state["request"]
    messages = state.get("messages", [])
    
    print(f"\n📝 Agent: Processing request: '{request}'")
    messages.append(f"User Request: {request}")
    
    # Use Claude to analyze the access request
    prompt = f"""Analyze this IT access request and extract:
1. User name
2. System/resource being requested
3. Action (grant or revoke)
4. Brief justification reasoning

Request: {request}

Respond in this format:
USER: [name]
SYSTEM: [system/resource]
ACTION: [grant or revoke]
REASONING: [brief reason]"""
    
    response = access_llm.invoke(prompt)
    claude_output = response.content.strip()
    print(f"🤖 Claude Analysis:\n{claude_output}\n")
    
    # Extract user
    user_match = re.search(r'USER:\s*(.+?)(?=\n|$)', claude_output)
    state["user"] = user_match.group(1).strip() if user_match else "Unknown"
    
    # Extract system
    sys_match = re.search(r'SYSTEM:\s*(.+?)(?=\n|$)', claude_output)
    state["system"] = sys_match.group(1).strip() if sys_match else "Unknown"
    
    # Extract action
    action_match = re.search(r'ACTION:\s*(.+?)(?=\n|$)', claude_output)
    action_str = action_match.group(1).strip().lower() if action_match else "grant"
    state["action"] = "grant" if "grant" in action_str else "revoke"
    
    # Extract reasoning
    reasoning_match = re.search(r'REASONING:\s*(.+?)(?=\n|$)', claude_output)
    state["reasoning"] = reasoning_match.group(1).strip() if reasoning_match else "Not provided"
    
    proposal = f"Proposing to {state['action']} '{state['system']}' access for user '{state['user']}'"
    print(f"🔧 Agent Proposal: {proposal}")
    print(f"   Reasoning: {state['reasoning']}")
    messages.append(f"Proposed Action: {proposal}")
    
    state["messages"] = messages
    
    return state

# Node 2: Wait for human approval (Simulated)
def human_approval_node(state: AccessControlState) -> AccessControlState:
    """Simulate human approval - in real systems, this would wait for user input"""
    messages = state.get("messages", [])
    
    user = state["user"]
    system = state["system"]
    action = state["action"]
    reasoning = state["reasoning"]
    
    print(f"\n⏸️  HUMAN-IN-THE-LOOP PAUSE")
    print(f"   Action: {action.upper()} '{system}' for user '{user}'")
    print(f"   Reasoning: {reasoning}")
    print(f"   Waiting for approval...")
    
    # Simulate approval (in real system, this would be user input)
    approval_response = "yes"
    
    if approval_response.lower() in ["yes", "y", "approve"]:
        state["approval_status"] = "APPROVED"
        messages.append(f"✅ Manager Approved: {action} access for {user}")
        print(f"   ✅ APPROVED by manager")
    else:
        state["approval_status"] = "REJECTED"
        messages.append(f"❌ Manager Rejected: {action} access for {user}")
        print(f"   ❌ REJECTED by manager")
    
    state["messages"] = messages
    return state

# Node 3: Execute approved action
def execute_access_node(state: AccessControlState) -> AccessControlState:
    """Execute the approved access control action"""
    messages = state.get("messages", [])
    
    if state["approval_status"] == "APPROVED":
        action = state["action"]
        user = state["user"]
        system = state["system"]
        
        print(f"\n⚙️  EXECUTING: {action}_access('{user}', '{system}')")
        
        if action == "grant":
            result = grant_access(user, system)
        else:
            result = revoke_access(user, system)
        
        state["execution_result"] = f"Access {result['status']}: {user} → {system}"
        messages.append(f"Execution Result: {state['execution_result']}")
    else:
        state["execution_result"] = "Action cancelled - approval rejected"
        messages.append(state["execution_result"])
        print(f"\n⛔ Action cancelled - approval was rejected")
    
    state["messages"] = messages
    return state

# Build the access control graph
access_graph_builder = StateGraph(AccessControlState)

# Add nodes
access_graph_builder.add_node("agent", access_control_agent)
access_graph_builder.add_node("approval", human_approval_node)
access_graph_builder.add_node("execute", execute_access_node)

# Define edges
access_graph_builder.add_edge(START, "agent")
access_graph_builder.add_edge("agent", "approval")  # PAUSE FOR HUMAN APPROVAL
access_graph_builder.add_edge("approval", "execute")
access_graph_builder.add_edge("execute", END)

# Compile the graph
access_control_graph = access_graph_builder.compile()

print("✓ Access Control Agent with Claude LLM & HITL built successfully!")
print("✓ Uses Claude for intelligent request analysis")
print("✓ Pauses for human approval before executing access changes")

✓ Access Control Agent with Claude LLM & HITL built successfully!
✓ Uses Claude for intelligent request analysis
✓ Pauses for human approval before executing access changes


In [8]:
### Exercise 3: Test Cases - Human-in-the-Loop Approval Workflow

print("=" * 80)
print("ACCESS CONTROL APPROVAL SYSTEM - HUMAN-IN-THE-LOOP DEMONSTRATION")
print("=" * 80)

# Test Case 1: Grant access request
print("\n🔷 SCENARIO 1: Grant Database Admin Access (APPROVED)")
print("-" * 80)

input_state_1 = {
    "request": "Give database admin access to user 'Kiran'",
    "user": "",
    "system": "",
    "action": "",
    "tool_call": {},
    "approval_status": "",
    "execution_result": "",
    "messages": []
}

result_1 = access_control_graph.invoke(input_state_1)

print("\n📊 Final State - Grant Access:")
print(f"  User: {result_1['user']}")
print(f"  System: {result_1['system']}")
print(f"  Action: {result_1['action']}")
print(f"  Approval Status: {result_1['approval_status']}")
print(f"  Execution Result: {result_1['execution_result']}")

print("\n📋 Message Log:")
for i, msg in enumerate(result_1['messages'], 1):
    print(f"  {i}. {msg}")

# Test Case 2: Revoke access request
print("\n\n🔷 SCENARIO 2: Revoke VPN Access (APPROVED)")
print("-" * 80)

input_state_2 = {
    "request": "Revoke VPN access from user 'Raj'",
    "user": "",
    "system": "",
    "action": "",
    "tool_call": {},
    "approval_status": "",
    "execution_result": "",
    "messages": []
}

result_2 = access_control_graph.invoke(input_state_2)

print("\n📊 Final State - Revoke Access:")
print(f"  User: {result_2['user']}")
print(f"  System: {result_2['system']}")
print(f"  Action: {result_2['action']}")
print(f"  Approval Status: {result_2['approval_status']}")
print(f"  Execution Result: {result_2['execution_result']}")

print("\n📋 Message Log:")
for i, msg in enumerate(result_2['messages'], 1):
    print(f"  {i}. {msg}")

print("\n" + "=" * 80)
print("✅ Exercise 3 Complete - Human-in-the-Loop Workflow Demonstrated!")
print("=" * 80)

print("\n🔍 KEY LEARNING POINTS:")
print("""
1. PAUSE MECHANISM:
   - Agent proposes tool call
   - System pauses at human approval node
   - Manager reviews the request

2. APPROVAL WORKFLOW:
   - Tool call is NOT executed immediately
   - Requires explicit human approval
   - Prevents unauthorized access changes

3. STATE TRACKING:
   - All messages are logged
   - Approval status is tracked
   - Execution result is recorded

4. REAL-WORLD APPLICATIONS:
   - IT access management
   - Financial transaction approvals
   - Sensitive data access requests
   - Enterprise security policies

5. MESSAGE FLOW:
   - User Request → Agent Proposal → HITL Pause → Approval/Rejection → Execution
""")

ACCESS CONTROL APPROVAL SYSTEM - HUMAN-IN-THE-LOOP DEMONSTRATION

🔷 SCENARIO 1: Grant Database Admin Access (APPROVED)
--------------------------------------------------------------------------------

📝 Agent: Processing request: 'Give database admin access to user 'Kiran''
🤖 Claude Analysis:
USER: Kiran
SYSTEM: Database Admin Access
ACTION: Grant
REASONING: Not provided in request

🔧 Agent Proposal: Proposing to grant 'Database Admin Access' access for user 'Kiran'
   Reasoning: Not provided in request

⏸️  HUMAN-IN-THE-LOOP PAUSE
   Action: GRANT 'Database Admin Access' for user 'Kiran'
   Reasoning: Not provided in request
   Waiting for approval...
   ✅ APPROVED by manager

⚙️  EXECUTING: grant_access('Kiran', 'Database Admin Access')
✓ Tool Executed: grant_access(Kiran, Database Admin Access)
  Status: granted

📊 Final State - Grant Access:
  User: Kiran
  System: Database Admin Access
  Action: grant
  Approval Status: APPROVED
  Execution Result: Access granted: Kiran → Database

---

## Summary & Comparison - All Exercises

### Exercise Overview

| Feature | Exercise 1 | Exercise 2 | Exercise 3 |
|---------|-----------|-----------|-----------|
| **Pattern** | Cyclic Graph | Memory-Based | Human-in-the-Loop |
| **Focus** | Multi-step tool execution | Context persistence | Approval workflow |
| **Key Component** | Tool loop | MemorySaver | Approval pause |
| **Real-world Use** | Financial calculations | Travel planning | IT access control |
| **State Flow** | LINEAR: A→B→C | CUMULATIVE: Turn1+Turn2+Turn3 | SEQUENTIAL: Propose→Pause→Execute |

### Key Concepts Demonstrated

#### Exercise 1: Cyclic Agent Graphs
- **Tool Loop**: Agent ↔ Tools ↔ Agent
- **Multi-step Execution**: Sequential tool calls
- **State Propagation**: Data flows through each tool node
- **Use Case**: Financial calculations, multi-step computations

#### Exercise 2: Memory-Based Chatbots
- **MemorySaver**: Persistent context storage
- **Thread ID**: Session tracking identifier
- **State Accumulation**: Information retained across turns
- **Use Case**: Conversational AI, personalized assistants

#### Exercise 3: Human-in-the-Loop
- **Approval Node**: Pauses before execution
- **Decision Point**: Requires human judgment
- **Risk Mitigation**: Prevents unauthorized actions
- **Use Case**: Enterprise systems, sensitive operations

### Common Patterns

```
Exercise 1: START → [Tool A] → [Tool B] → [Tool C] → END
            (Linear tool chain)

Exercise 2: Turn 1 + Turn 2 + Turn 3 + Turn 4 → Summary
            (Accumulate state across turns with memory)

Exercise 3: Propose → ⏸️ PAUSE ⏸️ → Approve/Reject → Execute
            (Decision point before execution)
```

### When to Use Each Pattern

| Scenario | Pattern | Why? |
|----------|---------|------|
| Calculate total cost and apply tax | Exercise 1 | Multiple sequential calculations |
| Customer support chatbot | Exercise 2 | Need to remember customer history |
| Approve high-value transactions | Exercise 3 | Requires human judgment |
| Multi-step data processing | Exercise 1 | Clear sequence of operations |
| Personalized recommendations | Exercise 2 | Requires long-term user preferences |
| Security approval workflows | Exercise 3 | Safety-critical decisions |

### Code Patterns to Remember

#### Pattern 1: Tool Loop (Exercise 1)
```python
graph_builder.add_edge(START, "tool_a")
graph_builder.add_edge("tool_a", "tool_b")
graph_builder.add_edge("tool_b", "tool_c")
graph_builder.add_edge("tool_c", END)
```

#### Pattern 2: Memory Saver (Exercise 2)
```python
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "session-001"}}
result = graph.invoke(state, config)
```

#### Pattern 3: HITL Approval (Exercise 3)
```python
def approval_node(state):
    # Propose action
    # Wait for human decision
    # Set approval_status
    return state

graph_builder.add_edge("agent", "approval")
graph_builder.add_edge("approval", "execute")
```

### Best Practices

1. **Always track messages** - Maintain audit trail of agent decisions
2. **Handle state propagation** - Ensure data flows correctly through nodes
3. **Plan your graph topology** - Linear, cyclic, or conditional based on use case
4. **Test with realistic inputs** - Validate agent behavior with real scenarios
5. **Consider edge cases** - What happens if inputs are invalid?

### Further Exploration

- Combine patterns: Use memory + approval for tracked decisions
- Add conditional routing: Different flows based on input
- Scale to production: Add error handling and retry logic
- Multi-agent systems: Coordinate multiple agents with messages